# 🇪🇬 Egyptian Currency Recognition System using YOLOv8
**Developer:** Ahmed Mohammed Saad El-Raggal  
**Domain:** Computer Vision & Financial Technology (FinTech)  

---

## 🛠️ Step 1: Project Initialization
In this stage, we initialize the environment and ensure all core libraries are correctly loaded. To avoid potential Windows DLL conflicts (WinError 127), we strictly enforce **CPU Execution Mode**.

====================================================================================================

import os
import sys
import cv2
import torch
from ultralytics import YOLO

# Force CPU mode to bypass CUDA/DLL dependency issues on Windows
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

print(f"✅ System Path: {sys.executable}")
print(f"✅ Torch Version: {torch.__version__}")
print(f"✅ Device: Running on CPU Mode")

In [2]:
%pip install ultralytics opencv-python matplotlib seaborn tqdm

import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ System Ready.")


[notice] A new release of pip available: 22.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ System Ready.


-------------------------

## 🧠 Step 2: Model Loading & Currency Mapping
We are utilizing a custom-trained **YOLOv8** model (`best.pt`). This step involves loading the weights and defining a dictionary to map the detected classes to their respective Egyptian Pound (EGP) values.

In [3]:
# Load the pre-trained weights from your training session
model = YOLO('best.pt') 

# Class-to-Value Mapping for Egyptian Banknotes (Front and Back)
currency_values = {
    '5_F': 5, '5_B': 5, '10_F': 10, '10_B': 10,
    '20_F': 20, '20_B': 20, '50_F': 50, '50_B': 50,
    '100_F': 100, '100_B': 100, '200_F': 200, '200_B': 200
}

print("✅ Model loaded successfully.")
print(f"✅ Total classes defined: {len(currency_values)}")

✅ Model loaded successfully.
✅ Total classes defined: 12


_____________


## 📸 Step 3: Live Detection & Real-time Inference
This is the core execution loop. The system captures live frames from the webcam, performs object detection, filters predictions with a confidence threshold, and overlays the total monetary value on the output stream.

In [ ]:
# Initialize Webcam
cap = cv2.VideoCapture(0)

print("🚀 Starting Detection Engine... Press 'q' to stop.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # Perform Inference (Enforce CPU and set confidence threshold)
    results = model.predict(frame, conf=0.6, device='cpu', verbose=False)
    
    total_amount = 0
    annotated_frame = results[0].plot()

    # Iterate through detections and accumulate total value
    for box in results[0].boxes:
        class_id = int(box.cls[0])
        class_name = model.names[class_id]
        total_amount += currency_values.get(class_name, 0)

    # UI Enhancement: Overlaying information
    cv2.rectangle(annotated_frame, (10, 10), (380, 70), (0, 0, 0), -1)
    cv2.putText(annotated_frame, f"Total: {total_amount} EGP", (20, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    # Display Output
    cv2.imshow("Egyptian Currency Recognition - V1.0", annotated_frame)
    
    # Exit condition
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

🚀 Starting Detection Engine... Press 'q' to stop.


ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device


[notice] A new release of pip available: 22.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cu121
     -------------                            0.8/2.4 GB 3.4 MB/s eta 0:08:08


______________________________________

## 🔊 Step 4: Audio Feedback Integration
In this step, we integrate a **Text-to-Speech (TTS)** engine using the `pyttsx3` library. This feature is crucial for accessibility, allowing the system to vocally announce the total amount detected. 

**Functionality:**
* Press **"S"** on your keyboard to trigger the voice assistant.
* The system will announce the total value in English (or Arabic if configured).

In [7]:
import os

def announce_total(amount):
    """
    Using Windows PowerShell directly to speak.
    No extra DLLs or complex libraries needed.
    """
    text = f"The total amount is {amount} Egyptian Pounds"
    # سطر واحد بيكلم محرك الويندوز مباشرة
    command = f'PowerShell -Command "Add-Type –AssemblyName System.Speech; (New-Object System.Speech.Synthesis.SpeechSynthesizer).Speak(\'{text}\')"'
    os.system(command)

print("✅ Alternative Voice Engine (PowerShell) is ready!")

✅ Alternative Voice Engine (PowerShell) is ready!


## 🎨 Step 5: Graphical User Interface (GUI) Design
To make the system user-friendly, we will design a professional **Dashboard**. Instead of a simple OpenCV window, we will use a structured layout.

**Planned Features:**
1. **Live Feed:** A large area to display the camera stream.
2. **Control Panel:** Buttons for starting/stopping the camera and triggering the voice assistant.
3. **Status Bar:** Real-time display of the detected currency and wallet total.
4. **Theme:** Modern Dark Mode for better visibility.

In [15]:
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageTk
import cv2
import os
from ultralytics import YOLO

class CurrencyApp:
    def __init__(self, window, window_title):
        self.window = window
        self.window.title(window_title)
        self.window.geometry("1000x800")
        self.window.configure(bg="#1e272e") # Dark Modern Theme

        # --- Variables ---
        self.total_in_view = 0
        self.model = YOLO('best.pt') 
        self.currency_values = {
            '5_F': 5, '5_B': 5, '10_F': 10, '10_B': 10,
            '20_F': 20, '20_B': 20, '50_F': 50, '50_B': 50,
            '100_F': 100, '100_B': 100, '200_F': 200, '200_B': 200
        }

        # --- UI Elements ---
        # Title
        self.header = tk.Label(window, text="EGP Currency Recognition Dashboard", 
                              font=("Segoe UI", 26, "bold"), bg="#1e272e", fg="#05c46b", pady=20)
        self.header.pack()

        # Main Layout (Video on left, Info on right)
        self.main_frame = tk.Frame(window, bg="#1e272e")
        self.main_frame.pack(padx=20, pady=10)

        # Video Feed
        self.canvas = tk.Canvas(self.main_frame, width=640, height=480, bg="#000", highlightthickness=2, highlightbackground="#05c46b")
        self.canvas.grid(row=0, column=0, padx=10)

        # Info Panel
        self.info_panel = tk.Frame(self.main_frame, bg="#2f3640", padx=20, pady=20)
        self.info_panel.grid(row=0, column=1, sticky="nsew")

        self.lbl_status = tk.Label(self.info_panel, text="CURRENT TOTAL", font=("Segoe UI", 14), bg="#2f3640", fg="#dcdde1")
        self.lbl_status.pack()

        self.amount_label = tk.Label(self.info_panel, text="0 EGP", font=("Segoe UI", 40, "bold"), bg="#2f3640", fg="#05c46b")
        self.amount_label.pack(pady=20)

        # Buttons
        self.btn_speak = tk.Button(self.info_panel, text="🔊 Speak Amount", width=15, command=self.speak_action,
                                 font=("Segoe UI", 12, "bold"), bg="#0fbcf9", fg="white", cursor="hand2")
        self.btn_speak.pack(pady=10)

        self.btn_exit = tk.Button(self.info_panel, text="🛑 Exit Program", width=15, command=self.close_app,
                                font=("Segoe UI", 12, "bold"), bg="#ff3f34", fg="white", cursor="hand2")
        self.btn_exit.pack(pady=10)

        # --- Camera Setup ---
        self.cap = cv2.VideoCapture(0)
        self.update_frame()
        self.window.protocol("WM_DELETE_WINDOW", self.close_app)
        self.window.mainloop()

    def speak_action(self):
        """Announce the total using PowerShell."""
        text = f"The total amount is {self.total_in_view} Egyptian Pounds"
        command = f'PowerShell -Command "Add-Type –AssemblyName System.Speech; (New-Object System.Speech.Synthesis.SpeechSynthesizer).Speak(\'{text}\')"'
        os.system(command)

    def update_frame(self):
        ret, frame = self.cap.read()
        if ret:
            # YOLO Inference
            results = self.model.predict(frame, conf=0.6, device='cpu', verbose=False)
            
            # Reset Total for current frame
            self.total_in_view = 0
            
            # Process Detections
            for box in results[0].boxes:
                class_id = int(box.cls[0])
                class_name = self.model.names[class_id]
                self.total_in_view += self.currency_values.get(class_name, 0)

            # Update Labels
            self.amount_label.config(text=f"{self.total_in_view} EGP")

            # Convert Frame for Tkinter
            annotated_frame = results[0].plot()
            rgb_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(rgb_frame)
            img_tk = ImageTk.PhotoImage(image=img)
            self.canvas.create_image(0, 0, image=img_tk, anchor=tk.NW)
            self.canvas.img_tk = img_tk 

        self.window.after(10, self.update_frame)

    def close_app(self):
        self.cap.release()
        self.window.destroy()

# --- RUN THE APP ---
# السطر ده هو اللي بيخلي الشباك يظهر فعلياً
root = tk.Tk()
app = CurrencyApp(root, "Currency Detector Pro V1.2")